# Import Data

In [ ]:
import sklearn as sk
import pandas as pd
import numpy as np
df = pd.read_parquet("basic_processed_data.parquet")

# Regression Target

In [ ]:
df['Severity'] = df['Severity'].astype(float) * 10
y = df['Severity']
X = df.copy().drop(columns=['Severity'])

# Part III.1: Perform 10-Fold Cross Validation 

## Advanced Preprocessing

In [ ]:
def add_road_type(f):
    HWY_RE = (
        r'\b[A-Z]{1,2}-\d+'                      # I-95, US-101, CA-99, WI-50
        r'|\b(?:Hwy|Fwy|Expy|Pkwy|Tpke|Turnpike|Highway|Freeway|Expressway|Interstate|Route|Rte|SR)\b'
    )
    
    LOCAL_RE = (
        r'\b(?:St|Street|Rd|Road|Ave|Avenue|Blvd|Boulevard|Ln|Lane|Dr|Drive|'
        r'Ct|Court|Pl|Place|Way|Pike|Cir|Circle|Trl|Trail|Ter|Terrace)\b'
    )   

    
    s = f['Street']
    f['road_type'] = np.select(
        [s.isna(), s.str.contains(HWY_RE, regex=True, na=False), s.str.contains(LOCAL_RE, regex=True, na=False)],
        [None, 'highway', 'local_street'],
        default='other'
    )
    
    return f

def weather_type(cond):
    if pd.isna(cond): return np.nan # or should i do "unknown"?
    elif ("Fair" in cond or cond == "Clear"): return "clear"
    elif ("Thunder" in cond or "T-Storm" in cond): return "storm"
    elif ("Rain" in cond or "Drizzle" in cond or "Precipitation" in cond or "Showers" in cond): return "rain"
    elif (cond == "Wintry Mix" or "Snow" in cond): return "snow"
    elif ("Cloudy" in cond or cond == "Cloudy" or "Clouds" in cond or "Overcast" in cond): 
        return "cloudy"
    elif ("Fog" in cond or "Mist" in cond or "Smoke" in cond or "Haze" in cond or "Dust" in cond or "Sand" in cond): 
        return "low-visibility"
    elif ("Sleet" in cond or "Hail" in cond or "Ice Pellets" in cond or "Tornado" in cond): 
        return "storm"
    elif ("Windy" in cond or "Squall" in cond): return "windy"
    else: return "cloudy"

def add_rush_hour(hour):
    if ((hour >= 7 and hour <= 9) or (hour >= 16 and hour <= 19)):
        return 1
    else: return 0

def advanced_preprocessing(fold):
    # Combine POI attributes into categories
    poi_type_groups = {
        "traffic_controlled": ['Traffic_Signal', 'Stop', 'Give_Way'],
        "junctions": ['Junction', 'Roundabout', 'No_Exit'],
        "traffic_calming": ['Bump', 'Traffic_Calming'],
        "pedestrians": ['Crossing', 'Station', 'Amenity']
    }
    
    F = fold.copy()
    for name, cols in poi_type_groups.items():
        F[name] = F[cols].fillna(False).astype(bool).any(axis=1).astype(int)
    
    F = F.drop(columns=poi_type_groups.get("traffic_controlled"))
    F = F.drop(columns=poi_type_groups.get("junctions"))
    F = F.drop(columns=poi_type_groups.get("traffic_calming"))
    F = F.drop(columns=poi_type_groups.get("pedestrians"))
    
    
    # Create road_type attribute from Street attribute
    F = add_road_type(F)
    F = F.drop(columns=['Street'])
    
    
    # Create time of day/lighting attribute from Sunrise_Sunset and Civil_Twilight
    F['tod_lighting'] = np.select(
        [F['Sunrise_Sunset'] == "Day", F['Civil_Twilight'] == "Day"],
        ['day', 'twilight'],
        default='night'
    )
    F = F.drop(columns=["Sunrise_Sunset", "Civil_Twilight","Nautical_Twilight", "Astronomical_Twilight"])
    
    
    # Create weather condition groups
    F['Weather_Condition'] = F['Weather_Condition'].apply(weather_type)
    
    
    # Create rush-hour flag from Start_Hour
    F['rush_hour'] = F['Start_Hour'].apply(add_rush_hour)
    
    return F




def listVals(f):
    vc = f["tod_lighting"].value_counts(dropna=False)
    print(len(vc), "unique values")
    print(vc.to_string())  






# Regression Techniques

In [ ]:
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold, cross_validate
from sklearn.pipeline import FunctionTransformer, Pipeline
from sklearn.preprocessing import OneHotEncoder


def perform_cv(Model, X, y):
    kf = KFold(n_splits=10, shuffle=True, random_state=1) 

    scoring_metrics = ["neg_mean_squared_error", "neg_root_mean_squared_error", "neg_mean_absolute_error", "r2"]

    cat_cols = make_column_selector(dtype_include=object)
    num_cols = make_column_selector(dtype_exclude=object)

    # don't impute for anything but linear regression???
    encode = ColumnTransformer([
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore"))]), 
        cat_cols),
        ("num", SimpleImputer(strategy="median"), num_cols),
    ])

    pipe = Pipeline([
        ("prep", FunctionTransformer(advanced_preprocessing)),
        ("encode", encode),
        ("model", Model),
    ])

    res = cross_validate(pipe, X, y, cv=kf, scoring=scoring_metrics)
    return res


## Dummy Regressor

In [ ]:
from sklearn.dummy import DummyRegressor

dr = DummyRegressor(strategy='mean')

Xs = X.sample(200_000, random_state=1)
ys = y.loc[Xs.index]

res = perform_cv(dr, Xs, ys)

print(res)

## Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression

Xs = X.sample(200_000, random_state=1)
ys = y.loc[Xs.index]

lr = LinearRegression()
res = perform_cv(lr, Xs, ys)

print(res)

## Regression Trees


In [ ]:
from sklearn.tree import DecisionTreeRegressor

dtr1 = DecisionTreeRegressor(random_state=False)
dtr2 = DecisionTreeRegressor(max_depth=5, random_state=False)

# res1 = perform_cv(dtr1)
# res2 = perform_cv(dtr2)


Xs = X.sample(200_000, random_state=1)
ys = y.loc[Xs.index]
res1 = perform_cv(dtr1, Xs, ys)
res2 = perform_cv(dtr2, Xs, ys)

print(res1)
print('\n')
print(res2)

## Random Forests

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rfr1 = RandomForestRegressor()
rfr2 = RandomForestRegressor(max_depth=5)

Xs = X.sample(200_000, random_state=1)
ys = y.loc[Xs.index]

res1 = perform_cv(rfr1, Xs, ys)
res2 = perform_cv(rfr2, Xs, ys)

print(res1)
print(res2)

{'fit_time': array([211.80011225, 269.88652611, 267.56937194, 170.8075254 ,
       263.22208166, 271.49069667, 275.60121679, 272.74151969,
       251.41645074, 249.18639016]), 'score_time': array([0.51328993, 0.51366544, 0.25804973, 0.27514338, 0.51038599,
       0.49560571, 0.52652287, 0.55372   , 0.47489643, 0.46544981]), 'test_neg_mean_squared_error': array([-12.14873096, -12.58849679, -12.50976314, -12.75188887,
       -12.08713328, -13.10963989, -11.96401308, -12.89431893,
       -12.58032536, -12.27568087]), 'test_neg_root_mean_squared_error': array([-3.48550297, -3.54802717, -3.53691435, -3.5709787 , -3.47665547,
       -3.62072367, -3.45890345, -3.59086604, -3.54687544, -3.50366677]), 'test_neg_mean_absolute_error': array([-1.33288857, -1.3653901 , -1.36474105, -1.38286826, -1.34310832,
       -1.41234762, -1.34766417, -1.38613746, -1.36748865, -1.34996632]), 'test_r2': array([0.225587  , 0.24506176, 0.22157867, 0.23329181, 0.2217096 ,
       0.21478322, 0.23631094, 0.22863308,

# Part III.2: Final Regression Model

In [25]:

dtr = DecisionTreeRegressor(max_depth=5, random_state=False)
res1 = perform_cv(dtr1, X, y)
print(res1)

{'fit_time': array([119.84732199, 112.09892893, 141.46399999, 148.21183562,
       156.08345127, 146.24858713, 146.78060794, 160.46379638,
       156.0857501 , 163.9300704 ]), 'score_time': array([1.37080503, 0.99031591, 1.38964915, 1.36206555, 1.35426426,
       1.35381317, 1.38748288, 1.41915059, 1.51888633, 1.43221426]), 'test_neg_mean_squared_error': array([-21.54894652, -21.74797928, -21.4359043 , -21.51062168,
       -21.5895173 , -21.50399128, -22.005826  , -21.57135458,
       -21.70628501, -21.48008701]), 'test_neg_root_mean_squared_error': array([-4.64208429, -4.66347288, -4.62989247, -4.63795447, -4.64645212,
       -4.63723962, -4.69103677, -4.64449724, -4.65900043, -4.63466148]), 'test_neg_mean_absolute_error': array([-1.2366803 , -1.24642188, -1.23359485, -1.23686479, -1.23944507,
       -1.23622908, -1.26483297, -1.23936566, -1.24771205, -1.23530346]), 'test_r2': array([-0.31176409, -0.32118867, -0.31248996, -0.30972984, -0.30058348,
       -0.3063659 , -0.32049858, -0.3